****
**Data Cleaning and Preprocessing**
****

In [1]:
#import library
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

**Setup and Load Raw Data**
****




In [3]:
from google.colab import drive
drive.mount('/content/drive')

Base_Path = '/content/drive/MyDrive/Colab Notebooks/DataScienceProject/Dataset/'
Clean     = Base_Path + 'cleaned/'
os.makedirs(Clean, exist_ok=True)

Mounted at /content/drive


In [ ]:
train_transaction = pd.read_csv(Base_Path + 'train_transaction.csv')
train_identity = pd.read_csv(Base_Path + 'train_identity.csv')
test_transaction = pd.read_csv(Base_Path + 'test_transaction.csv')
test_identity = pd.read_csv(Base_Path + 'test_identity.csv')

**Basic Info**
****

In [ ]:
print("Raw shapes:")
print(f"  Train Transaction : {train_transaction.shape}")
print(f"  Train Identity    : {train_identity.shape}")
print(f"  Test  Transaction : {test_transaction.shape}")
print(f"  Test  Identity    : {test_identity.shape}")

Raw shapes:
  Train Transaction : (590540, 394)
  Train Identity    : (144233, 41)
  Test  Transaction : (506691, 393)
  Test  Identity    : (141907, 41)


In [ ]:
print("\n--- Train Transaction Info ---")
train_transaction.info()
print("\n--- Train Transaction Head ---")
print(train_transaction.head())

print("\n--- Train Identity Info ---")
train_identity.info()
print("\n--- Train Identity Head ---")
print(train_identity.head())

print("\n--- Test Transaction Info ---")
test_transaction.info()
print("\n--- Test Transaction Head ---")
print(test_transaction.head())

print("\n--- Test Identity Info ---")
test_identity.info()
print("\n--- Test Identity Head ---")
print(test_identity.head())


--- Train Transaction Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 394 entries, TransactionID to V339
dtypes: float64(376), int64(4), object(14)
memory usage: 1.7+ GB

--- Train Transaction Head ---
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ... V330  V331  V332  V333  V334 V335  \
0    NaN  150.0    discover  142.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
1  404.0  150.0  mastercard  102.0  ...  NaN   NaN   NaN   NaN   NaN  NaN   
2  490.0  150.0        visa  166.0  ...  NaN   NaN   Na

**Merging**
****

In [ ]:
train = train_transaction.merge(train_identity, how='left', on='TransactionID')
test = test_transaction.merge(test_identity, how='left', on='TransactionID')

In [ ]:
print(f"\nAfter merge:")
print(f"  Train : {train.shape}")
print(f"  Test  : {test.shape}")


After merge:
  Train : (590540, 434)
  Test  : (506691, 433)


In [ ]:
print("\n--- Train After Merging ---")
train.info()
print("\n--- Train After Merging ---")
print(train.head())


--- Train After Merging ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Columns: 434 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(4), object(31)
memory usage: 1.9+ GB

--- Train After Merging ---
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD  card1  \
0        2987000        0          86400            68.5         W  13926   
1        2987001        0          86401            29.0         W   2755   
2        2987002        0          86469            59.0         W   4663   
3        2987003        0          86499            50.0         W  18132   
4        2987004        0          86506            50.0         H   4497   

   card2  card3       card4  card5  ...                id_31  id_32  \
0    NaN  150.0    discover  142.0  ...                  NaN    NaN   
1  404.0  150.0  mastercard  102.0  ...                  NaN    NaN   
2  490.0  150.0        visa  166.0  ...                  NaN    NaN   
3 

In [ ]:
print("\n--- Test After Merging ---")
test.info()
print("\n--- Test After Merging ---")
print(test.head())


--- Test After Merging ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 506691 entries, 0 to 506690
Columns: 433 entries, TransactionID to DeviceInfo
dtypes: float64(399), int64(3), object(31)
memory usage: 1.6+ GB

--- Test After Merging ---
   TransactionID  TransactionDT  TransactionAmt ProductCD  card1  card2  \
0        3663549       18403224           31.95         W  10409  111.0   
1        3663550       18403263           49.00         W   4272  111.0   
2        3663551       18403310          171.00         W   4476  574.0   
3        3663552       18403310          284.95         W  10989  360.0   
4        3663553       18403317           67.95         W  18018  452.0   

   card3       card4  card5  card6  ...  id-31  id-32  id-33  id-34 id-35  \
0  150.0        visa  226.0  debit  ...    NaN    NaN    NaN    NaN   NaN   
1  150.0        visa  226.0  debit  ...    NaN    NaN    NaN    NaN   NaN   
2  150.0        visa  226.0  debit  ...    NaN    NaN    NaN    NaN  

In [ ]:
del train_transaction, train_identity, test_transaction, test_identity

**Memory Reduction**
****

In [ ]:
def reduce_memory(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                else:
                    df[col] = df[col].astype(np.int32)
            else:
                df[col] = df[col].astype(np.float32)
    return df

In [ ]:
train = reduce_memory(train)
print(f"\nAfter memory reduction:")
print(f"  Train Memory Usage : {train.memory_usage().sum() / 1024**2:.2f} MB")


After memory reduction:
  Train Memory Usage : 1044.70 MB


In [ ]:
test = reduce_memory(test)
print(f"\nAfter memory reduction:")
print(f"  Test Memory Usage : {test.memory_usage().sum() / 1024**2:.2f} MB")

**Standardize Column Naming**
****

In [ ]:
rename_map = {col: col.replace('-', '_')
              for col in test.columns if str(col).lower().startswith('id-')}
test.rename(columns=rename_map, inplace=True)

In [ ]:
print(f"\nRenamed {len(rename_map)} hyphen-named columns in test to underscore format.")

**Drop High Percentage Missingness Column**
****

In [ ]:
threshold = 0.9
missing_ratio = train.isnull().sum() / len(train)
cols_to_drop  = missing_ratio[missing_ratio > threshold].index.tolist()

In [ ]:
train.drop(columns=cols_to_drop, inplace=True)
test.drop(columns=[c for c in cols_to_drop if c in test.columns], inplace=True)

In [ ]:
print(f"\nDropped {len(cols_to_drop)} columns with >{int(threshold*100)}% missing values.")
print(f"  Train : {train.shape}")
print(f"  Test  : {test.shape}")

**Define Feature Groups**
****

In [ ]:
#DeviceType, DeviceInfo and 'D' all of them start with D but in different category
v_cols      = [c for c in train.columns if c.startswith('V')]
c_cols      = [c for c in train.columns if c.startswith('C')]
d_cols      = [c for c in train.columns if c.startswith('D')
               and c not in ('DeviceType', 'DeviceInfo')]
m_cols      = [c for c in train.columns if c.startswith('M')]
id_cols     = sorted([c for c in train.columns if str(c).lower().startswith('id')])
id_num_cols = [c for c in id_cols if train[c].dtype != 'object']
id_cat_cols = [c for c in id_cols if train[c].dtype == 'object']

In [ ]:
print(f"\nFeature groups:")
print(f"  V cols           : {len(v_cols)}")
print(f"  C cols           : {len(c_cols)}")
print(f"  D cols (numeric) : {len(d_cols)}")
print(f"  M cols           : {len(m_cols)}")
print(f"  ID cols total    : {len(id_cols)}")
print(f"    └ numeric      : {len(id_num_cols)}")
print(f"    └ categorical  : {len(id_cat_cols)}")

**Handle Missing Values by Feature Group**
****

In [ ]:
#Handling V Columns
for col in v_cols:
  train[col] = train[col].fillna(-999)
  test[col] = test[col].fillna(-999)

print(f"\n[V cols] Filled {len(v_cols)} columns with -999.")

In [ ]:
#Handling C Columns
for col in c_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

print(f"[C cols] Filled {len(c_cols)} columns with 0.")

In [ ]:
#Handling D Columns
d_medians      = {}
d_all_nan_cols = []

for col in d_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce')
    test[col]  = pd.to_numeric(test[col],  errors='coerce')
    if train[col].isna().all():
        train[col] = train[col].fillna(0)
        test[col]  = test[col].fillna(0)
        d_all_nan_cols.append(col)
    else:
        median_val       = train[col].median()
        d_medians[col]   = median_val
        train[col]       = train[col].fillna(median_val)
        test[col]        = test[col].fillna(median_val)

print(f"[D cols] Filled {len(d_cols) - len(d_all_nan_cols)} with median, "
      f"{len(d_all_nan_cols)} all-NaN cols filled with 0.")

In [ ]:
#Handling M Columns
for col in m_cols:
    train[col] = train[col].fillna('Unknown')
    test[col]  = test[col].fillna('Unknown')

print(f"[M cols] Filled {len(m_cols)} columns with 'Unknown'.")

In [ ]:
#Handling ID Columns
for col in id_cat_cols:
    train[col] = train[col].fillna('unknown')
    test[col]  = test[col].fillna('unknown')

print(f"[ID cat] Filled {len(id_cat_cols)} columns with 'unknown'.")

In [ ]:
#preventing numeric column filled with unknown
for col in id_num_cols:
    train[col] = pd.to_numeric(train[col], errors='coerce').fillna(-999)
    test[col]  = pd.to_numeric(test[col],  errors='coerce').fillna(-999)
print(f"[ID num] Filled {len(id_num_cols)} columns with -999.")

In [ ]:
#Checking for ID
remaining_id_train = train[id_cols].isnull().sum().sum()
remaining_id_test  = test[id_cols].isnull().sum().sum()
status = "PASSED" if remaining_id_train == 0 and remaining_id_test == 0 else "FAILED"
print(f"\n[ID check] {status} — nulls: train={remaining_id_train}, test={remaining_id_test}")

**Handle Device Type and Device Info**
****

In [ ]:
_train_tx = pd.read_csv(Base_Path + 'train_transaction.csv', usecols=['TransactionID'])
_train_id = pd.read_csv(Base_Path + 'train_identity.csv',
                        usecols=['TransactionID', 'DeviceType', 'DeviceInfo'])
_test_tx  = pd.read_csv(Base_Path + 'test_transaction.csv',  usecols=['TransactionID'])
_test_id  = pd.read_csv(Base_Path + 'test_identity.csv',
                        usecols=['TransactionID', 'DeviceType', 'DeviceInfo'])

In [ ]:
_train_device = _train_tx.merge(_train_id, on='TransactionID', how='left')
_test_device  = _test_tx.merge(_test_id,   on='TransactionID', how='left')

In [ ]:
train['DeviceType'] = _train_device['DeviceType'].values
train['DeviceInfo'] = _train_device['DeviceInfo'].values
test['DeviceType']  = _test_device['DeviceType'].values
test['DeviceInfo']  = _test_device['DeviceInfo'].values

In [ ]:
del _train_tx, _train_id, _test_tx, _test_id, _train_device, _test_device

In [ ]:
# Standardise and fill
for col in ['DeviceType', 'DeviceInfo']:
    train[col] = train[col].fillna('unknown').astype(str).str.strip().str.lower()
    test[col]  = test[col].fillna('unknown').astype(str).str.strip().str.lower()

print(f"  DeviceType unique : {sorted(train['DeviceType'].unique())}")
print(f"  DeviceInfo sample : {list(train['DeviceInfo'].unique()[:4])}")

**Handle Remaining Categorical Columns**
****

In [ ]:
already_handled_cat = set(m_cols + id_cat_cols +
                          ['DeviceType', 'DeviceInfo',
                           'R_emaildomain', 'P_emaildomain', 'card4', 'card6'])

remaining_cat = [c for c in train.select_dtypes(include='object').columns
                 if c not in already_handled_cat]

In [ ]:
for col in remaining_cat:
    train[col] = train[col].fillna('unknown')
    test[col]  = test[col].fillna('unknown')

In [ ]:
for col in ['R_emaildomain', 'P_emaildomain', 'card4', 'card6']:
    if col in train.columns:
        train[col] = train[col].fillna('unknown')
        test[col]  = test[col].fillna('unknown')

In [ ]:
print(f"\n[Cat cols] Filled remaining categoricals: {remaining_cat + ['R_emaildomain','P_emaildomain','card4','card6']}")

**Handle Remaining Numeric Columns**
****

In [ ]:
num_null_cols = ['dist1', 'addr1', 'addr2', 'card2', 'card3', 'card5']

for col in num_null_cols:
    if col in train.columns:
        median_val = train[col].median()
        train[col] = train[col].fillna(median_val)
        test[col]  = test[col].fillna(median_val)
        print(f"  {col:<10} filled with median: {median_val}")

**Drop Test only ID Columns**
****

In [ ]:
train_id_set  = set([c for c in train.columns if str(c).lower().startswith('id')])
test_id_set   = set([c for c in test.columns  if str(c).lower().startswith('id')])
test_only_ids = sorted(test_id_set - train_id_set)

In [ ]:
if test_only_ids:
    test.drop(columns=test_only_ids, inplace=True)
    print(f"\nDropped {len(test_only_ids)} test-only ID columns: {test_only_ids}")
else:
    print("\nNo test-only ID columns to drop.")

**Log-Transform TransactionAmt**
****

In [ ]:
train['TransactionAmt_log'] = np.log1p(train['TransactionAmt'])
test['TransactionAmt_log']  = np.log1p(test['TransactionAmt'])
print("\n[Feature] TransactionAmt_log created.")

In [ ]:
#Missing value indicator is flags for high-missingness columns
key_missing_cols = ['DeviceType', 'DeviceInfo',
                    'id_30', 'id_31', 'id_33',
                    'R_emaildomain', 'P_emaildomain']

In [ ]:
print("\n[Feature] Missing-value indicators:")
for col in key_missing_cols:
    if col not in train.columns:
        continue
    flag_col = f'{col}_was_missing'
    if train[col].dtype == 'object':
        train[flag_col] = (train[col] == 'unknown').astype(np.int8)
        test[flag_col]  = (test[col]  == 'unknown').astype(np.int8)
    else:
        train[flag_col] = (train[col] == -999).astype(np.int8)
        test[flag_col]  = (test[col]  == -999).astype(np.int8)
    pct = train[flag_col].mean() * 100
    print(f"  {flag_col:<35} — {pct:.1f}% were missing")

**Standardise All Categorical COlumns (Optional for CatBoost)**
****

In [ ]:
final_cat_cols = train.select_dtypes(include='object').columns.tolist()

In [ ]:
for col in final_cat_cols:
    train[col] = train[col].astype(str).str.strip().str.lower()
    test[col]  = test[col].astype(str).str.strip().str.lower()

print(f"\nStandardised {len(final_cat_cols)} categorical columns to lowercase stripped strings.")

**Column Alignment Check**
****

In [ ]:
train_cols   = set(train.columns) - {'isFraud'}
test_cols    = set(test.columns)
only_in_train = sorted(train_cols - test_cols)
only_in_test  = sorted(test_cols - train_cols)

In [ ]:
print("-- Alignment Check --")
print(f"Only in train (excl. isFraud) : {only_in_train if only_in_train else '[]'}")
print(f"Only in test                  : {only_in_test  if only_in_test  else '[]'}")
if not only_in_train and not only_in_test:
    print("PASSED — train and test columns are aligned.")
else:
    print("WARNING — column mismatch, investigate before modelling.")

**Final Summary**

In [ ]:
print("\n -- FINAL CLEANING SUMMARY --")
print(f"Train shape           : {train.shape}")
print(f"Test  shape           : {test.shape}")
print(f"Nulls in train        : {train.isnull().sum().sum()}")
print(f"Nulls in test         : {test.isnull().sum().sum()}")
print(f"Fraud ratio preserved : {train['isFraud'].mean():.4f}")
print(f"isFraud not in test   : {'isFraud' not in test.columns}")
print(f"Train memory usage    : {train.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Test  memory usage    : {test.memory_usage(deep=True).sum()  / 1e6:.1f} MB")

In [ ]:
#Dtype alignment check between train and test
dtype_mismatches = []
for col in train.columns:
    if col == 'isFraud':
        continue
    if col in test.columns:
        if train[col].dtype != test[col].dtype:
            dtype_mismatches.append((col, train[col].dtype, test[col].dtype))

if len(dtype_mismatches) == 0:
    print("PASSED - All dtypes match between train and test")
else:
    print(f"WARNING - {len(dtype_mismatches)} dtype mismatches found:")
    for col, t_dtype, te_dtype in dtype_mismatches:
        print(f"  {col:<20} train={t_dtype}, test={te_dtype}")
    for col, t_dtype, _ in dtype_mismatches:
        try:
            test[col] = test[col].astype(t_dtype)
        except Exception as e:
            print(f"  Could not cast {col}: {e}")
    print(f"\nForce-cast {len(dtype_mismatches)} test columns to match train dtypes.")

    # Re-verify
    still_broken = [(c, train[c].dtype, test[c].dtype)
                    for c in train.columns if c != 'isFraud'
                    and c in test.columns and train[c].dtype != test[c].dtype]
    if not still_broken:
        print("PASSED - All dtypes now match after force-cast.")
    else:
        print(f"STILL BROKEN - {len(still_broken)} columns unresolved: {still_broken}")

**Save Cleaned Files**
****

In [ ]:
train.to_csv(Clean + 'train_cleaned.csv', index=False)
test.to_csv(Clean  + 'test_cleaned.csv',  index=False)

In [ ]:
print(f"\nSaved → {Clean}train_cleaned.csv")
print(f"Saved → {Clean}test_cleaned.csv")
print("Done.")